# AMR Project - Step 3: Regularized Linear Baselines (NCBI Dataset)

This notebook trains L1 (Lasso) and L2 (Ridge) regularized Logistic Regression classifiers as baseline models.

### Objectives:
1. **Split Dataset**: Perform an 80/20 train/test stratified split on `preprocessed_data.csv`.
2. **L1 Regularization (Lasso)**: Train an L1 model to perform automated feature selection (shrinking coefficients of non-informative columns to 0).
3. **L2 Regularization (Ridge)**: Train an L2 model to prevent overfitting.
4. **Evaluation**: Compute Accuracy, Precision, Recall, F1 score, and inspect coefficients.

In [18]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 1. Load Preprocessed Data

In [19]:
preprocessed_path = "../data/processed/preprocessed_data.csv"
print(f"Loading data from {preprocessed_path}...")
df = pd.read_csv(preprocessed_path, low_memory=False)
print(f"Dataset shape: {df.shape}")

Loading data from ../data/processed/preprocessed_data.csv...
Dataset shape: (401831, 141)


## 2. Train/Test Stratified Split

In [20]:
X = df.drop(columns=['Target'])
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")

Train set: 321464 rows | Test set: 80367 rows


## 3. Train L1 (Lasso) Logistic Regression

In [21]:
l1_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42, max_iter=300)
print("Training L1 Regularized Logistic Regression...")
l1_model.fit(X_train, y_train)
print("Model trained.")

Training L1 Regularized Logistic Regression...
Model trained.


### L1 Evaluation

In [22]:
y_pred_l1 = l1_model.predict(X_test)

print("Accuracy: ", accuracy_score(y_test, y_pred_l1))
print("Precision:", precision_score(y_test, y_pred_l1))
print("Recall:   ", recall_score(y_test, y_pred_l1))
print("F1 Score: ", f1_score(y_test, y_pred_l1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l1))

non_zero_coefs = np.sum(l1_model.coef_[0] != 0)
print(f"\nLasso kept {non_zero_coefs} non-zero coefficients out of {X_train.shape[1]} features.")

Accuracy:  0.9566488732937648
Precision: 0.9198251505943814
Recall:    0.897062646217832
F1 Score:  0.9083013107332737

Confusion Matrix:
[[59628  1504]
 [ 1980 17255]]

Lasso kept 130 non-zero coefficients out of 140 features.


## 4. Train L2 (Ridge) Logistic Regression

In [23]:
l2_model = LogisticRegression(penalty='l2', solver='liblinear', C=1.0, random_state=42, max_iter=300)
print("Training L2 Regularized Logistic Regression...")
l2_model.fit(X_train, y_train)
print("Model trained.")

Training L2 Regularized Logistic Regression...
Model trained.


### L2 Evaluation

In [24]:
y_pred_l2 = l2_model.predict(X_test)

print("Accuracy: ", accuracy_score(y_test, y_pred_l2))
print("Precision:", precision_score(y_test, y_pred_l2))
print("Recall:   ", recall_score(y_test, y_pred_l2))
print("F1 Score: ", f1_score(y_test, y_pred_l2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l2))

Accuracy:  0.956586658703199
Precision: 0.9193565569404496
Recall:    0.8973225890304133
F1 Score:  0.908205951222079

Confusion Matrix:
[[59618  1514]
 [ 1975 17260]]


## 5. Save Models

In [25]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

with open(os.path.join(models_dir, "logistic_l1.pkl"), 'wb') as f:
    pickle.dump(l1_model, f)
    
with open(os.path.join(models_dir, "logistic_l2.pkl"), 'wb') as f:
    pickle.dump(l2_model, f)
    
print("Baseline models saved successfully.")

Baseline models saved successfully.
